# RAFT · Stage 4 — Baseline vs RAFT evaluation

Runs the same question set against the **baseline** (`gpt-4.1`) and the **RAFT fine-tuned model**, scores
groundedness, retrieval quality and relevance, and captures tokens per investigation, latency and
cost per 1,000 investigations. Emits JSON + a chart under `eval/results/` — the numbers the app's
Model Quality tab (WS-9) renders.

**Runtime:** same scale as Stage 1. Always reports baseline **and** tuned together; a tuned score
alone is not evidence. Delegates scoring to `eval/harness.py` so results are reproducible from one
command outside the notebook too.

In [ ]:
baseline_model = "gpt-4.1"
student_deployment = "raft-student"
eval_questions_path = "data/eval_questions.jsonl"
corpus_manifest = "../../fabric/lakehouse/corpus/manifest.yaml"
eval_results_dir = "eval/results"

In [ ]:
import sys, pathlib, json, time
sys.path.insert(0, str(pathlib.Path("eval").resolve()))
from harness import run_comparison, write_results  # eval/harness.py (WS-7)
t0 = time.time()
print(f"Stage 4 · baseline={baseline_model} vs fine-tuned deployment={student_deployment}")

In [ ]:
# run_comparison returns baseline & tuned metrics plus economics; offline it uses deterministic stubs.
results = run_comparison(
    eval_questions_path=eval_questions_path,
    corpus_manifest=corpus_manifest,
    baseline_model=baseline_model,
    student_deployment=student_deployment,
)
out = write_results(results, eval_results_dir)
print(json.dumps(results["summary"], indent=2))
print(f"Stage 4 done in {time.time()-t0:.0f}s. Wrote {out}")

In [ ]:
# Optional chart (baseline vs RAFT) for the slide.
try:
    import matplotlib.pyplot as plt
    s = results["summary"]
    metrics = ["groundedness", "retrieval_quality", "relevance"]
    x = range(len(metrics))
    plt.figure(figsize=(6, 3.5))
    plt.bar([i - 0.2 for i in x], [s["baseline"][m] for m in metrics], width=0.4, label="baseline")
    plt.bar([i + 0.2 for i in x], [s["raft"][m] for m in metrics], width=0.4, label="RAFT")
    plt.xticks(list(x), metrics); plt.ylim(0, 1); plt.legend(); plt.title("Baseline vs RAFT")
    plt.tight_layout(); plt.savefig(pathlib.Path(eval_results_dir) / "baseline_vs_raft.png", dpi=120)
    plt.show()
except Exception as e:
    print(f"chart skipped: {e}")